In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
quants = pd.read_csv("analysis/quants_all.csv", index_col=0)
quants["question_type"] = quants["question_type"].str.rsplit("_", n=1).str[0]
quants

In [ ]:
df = pd.concat(
    {
        "TS Evol-Instruct": pd.read_csv(
            "analysis/chatts_sft_evol_instruct_train.csv", index_col=0
        ),
        "(Tune-)OpenSQA": pd.read_csv(
            "analysis/llasa_open_sqa_v2_and_tune.csv", index_col=0
        ),
        "QuAnTS": quants,
    },
).droplevel(1, axis="index")[["question_type", "answer_type"]]
df

In [ ]:
df["question_type"].value_counts(normalize=True, dropna=False) * 100

In [ ]:
df

In [ ]:
# --- 3. Plotting ---
# Set the visual style of the plots
sns.set_style("whitegrid")

# Create a figure and a set of subplots. The size and number of plots are adjusted automatically.
num_datasets = df.index.nunique()
fig, axes = plt.subplots(num_datasets, 1, figsize=(8, 4 * num_datasets), squeeze=False)
axes = axes.squeeze(-1)

# Loop through the datasets and create a plot for each
for i, (title, df_sub) in enumerate(df.groupby(level=0)):
    # Calculate the frequency of each class/category
    class_counts = df_sub["question_type"].fillna("<?>").value_counts()

    # Create the horizontal bar plot on the appropriate subplot
    sns.barplot(x=class_counts.values, y=class_counts.index, ax=axes[i], orient="h")

    # Customize the plot with a title and labels
    axes[i].set_title(title, fontsize=14)
    axes[i].set_xlabel("Count", fontsize=12)
    axes[i].set_ylabel("Question Type", fontsize=12)

    # Add the exact count as a text label on each bar for clarity
    for index, value in enumerate(class_counts):
        axes[i].text(value, index, f" {value}", va="center", fontsize=10)

# Adjust the layout to prevent titles and labels from overlapping
plt.tight_layout()